# 21 — PharmaLens Fact Data Intelligence

**FACT = real evidence. SYNTHETIC = demo/testing/marketing only.**

This is a shared data/evidence foundation. It must reuse existing ingestion and canonicalization components rather than duplicate them.

In [ ]:
from pathlib import Path
import sys, pandas as pd
PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'data').exists(): PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT/'src'))
from fact_data_intelligence import *
RAW=PROJECT_ROOT/'data'/'raw'
print('Project:',PROJECT_ROOT)
print('FACT:',RAW/'fact_raw_data')
print('SYNTHETIC:',RAW/'synthetic_raw_data')

## 1. Raw data inventory

In [ ]:
inventory=discover_files(RAW)
if inventory.empty: print('No supported files found')
else: display(inventory[['file_name','data_type','extension','size_mb']])

## 2. FACT profiling only

The notebook discovers schemas; it does not assume exact IQVIA column names and does not fabricate mappings.

In [ ]:
fact_files=[] if inventory.empty else [Path(p) for p in inventory.loc[inventory.data_type=='FACT','path']]
for p in fact_files:
    print('\n###',p.name)
    try:
        if p.suffix.lower() in {'.xlsx','.xls'}:
            xl=pd.ExcelFile(p)
            for sh in xl.sheet_names:
                df=pd.read_excel(p,sheet_name=sh)
                print('SHEET:',sh, profile_table(df))
        else:
            df=read_table(p)
            print(profile_table(df))
    except Exception as e: print('SCAN ERROR:',e)

## 3. FACT quality checks

In [ ]:
for p in fact_files:
    try:
        if p.suffix.lower() in {'.xlsx','.xls'}:
            for sh in pd.ExcelFile(p).sheet_names:
                print(p.name,sh,validate_table(pd.read_excel(p,sheet_name=sh)))
        else: print(p.name,validate_table(read_table(p)))
    except Exception as e: print(p.name,'ERROR',e)

## 4. Provenance contract

In [ ]:
for p in fact_files: print(provenance(p))

## 5. Synthetic guardrail

In [ ]:
try:
    guard_real_business_data('SYNTHETIC')
    print('FAIL: synthetic was not blocked')
except ValueError as e:
    print('PASS:',e)
guard_real_business_data('FACT')
print('PASS: FACT accepted')

## Downstream contract

`FACT → Canonical Data → Analytics Engines → Structured Intelligence → Decision Engine → AI Copilot`

Do not duplicate Market Intelligence, Forecasting, Target Planning, Market Access, Medical Affairs, Event Intelligence, or Finance here.